# Introduction

## Objectif métier

L'objectif de cette analyse est de mieux comprendre les entreprises adhérentes à l'UDM et d'identifier les caractéristiques associées à une adhésion durable (3 ans ou plus).  

Cette analyse permettra de :
- dresser le portrait des adhérents ;
- comprendre leur niveau d'engagement avec l'UDM ;
- mesurer leur poids économique ;
- identifier les différences entre les adhérents durables et les autres entreprises.

## Plan de l'analyse

- **0. Variable cible : adhésion durable**
- **1. Qui sont les adhérents ?**
  - 1.1 Répartition sectorielle
  - 1.2 Relation historique avec l'UDM
  - 1.3 Structure organisationnelle
  - 1.4 Présence des différents départements
  - 1.5 Nombre de contacts identifiés par entreprise
  - 1.6 Indicateurs temporels des contacts
  - 1.7 Budget marketing
- **2. Comment les membres interagissent avec l'UDM ?**
  - 2.1 Comment nos adhérents interagissent-ils avec nos emails ?
  - 2.2 Comment nos adhérents interagissent-ils avec nos évènements ?
  - 2.3 Appétence à recevoir les communications

## Chargement des données

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.analyse_univariee import analyse_univariee

# données agrégées au niveau adhérent
data = pd.read_excel("../data/base_adherents.xlsx")
print(f"Dimensions : {data.shape[0]:,} lignes × {data.shape[1]} colonnes")

# données agrégées au niveau membre
contacts = pd.read_excel("../data/base_membres_propre.xlsx")
print(f"Dimensions : {contacts.shape[0]:,} lignes × {contacts.shape[1]} colonnes")


# 0. Variable cible : adhésion durable
La variable target indique si une entreprise est adhérente depuis 3 ans ou plus :
- 1 : adhésion de 3 ans ou plus
- 0 : adhésion inférieure à 3 ans

In [ ]:
analyse_univariee(data, 'target')

La population étudiée est très majoritairement composée d'entreprises adhérentes depuis au moins trois ans.  
Plus de 8 entreprises sur 10 appartiennent à cette catégorie, ce qui témoigne d'une forte fidélisation des adhérents présents dans la base. À l'inverse, les adhésions courtes représentent une part relativement limitée des entreprises observées.

# 1. Qui sont les adhérents ?
Observons le profil des entreprises présentes dans la base.

## 1.1 Répartition sectorielle

On effectue un  regroupement de secteurs selon le mapping ci-dessous.

In [ ]:
sector_mapping = {
    "Alimentaire": "Grande consommation",
    "FMCG": "Grande consommation",
    "Boissons alcoolisées": "Grande consommation",
    "Banque Assurance": "Banque Assurance",
    "Services": "Services",
    "Hôtellerie, loisirs et voyages": "Services",
    "Distribution et eCommerce": "Distribution et eCommerce",
    "Mode, beauté, luxe": "Mode, beauté, luxe",
    "Automobile et mobilité": "Industrie & Mobilité",
    "Énergie et environnement": "Industrie & Mobilité",
    "Industrie": "Industrie & Mobilité",
    "Médias": "Technologie & Médias",
    "Télécommunications": "Technologie & Médias",
    "High-tech": "Technologie & Médias",
    "Santé": "Santé",
    "BtoB": "BtoB",
    "Secteur public": "Secteur public",
}

data["sector_group"] = (
    data["Organisation - Sector of Activities"]
    .astype(str)
    .str.strip()
    .map(sector_mapping)
)

data["sector_group"] = data["sector_group"].fillna("Autre")
analyse_univariee(data, 'sector_group')

L'UDM regroupe des entreprises issues d'un large éventail de secteurs d'activité. 

Cette répartition montre que l'UDM possède un positionnement fortement transversal et attire des organisations aux problématiques marketing variées.  

Même si certains secteurs sont davantage représentés, aucun secteur ne domine totalement la base. Cela suggère que les enjeux portés par l'UDM concernent un spectre relativement large d'entreprises.  

## 1.2 Relation historique avec l'UDM

### Ancienneté des adhésions

In [ ]:
analyse_univariee(data, 'GROUPE - Année adhésion *')

La variable année d'adhésion met en évidence une forte diversité dans l'ancienneté des entreprises membres de l'UDM.  

Les statistiques descriptives montrent :
- année moyenne d'adhésion : 2010
- médiane : 2015
- minimum observé : 1994
- maximum observé : 2026

**Point de vigilance**  
Une limite importante doit être prise en compte dans l'interprétation :
toutes les entreprises adhérentes avant 1994 ont été regroupées sous la valeur 1994.
Ainsi, le pic observé en 1994 ne correspond pas nécessairement à une vague massive d'adhésions cette année-là. Il reflète en réalité l'existence d'entreprises historiques présentes avant le début de l'information disponible.  

Ce résultat suggère que l'UDM bénéficie d'un noyau d'entreprises très anciennes, auquel viennent progressivement s'ajouter de nouveaux adhérents.  

L'organisation combine ainsi à la fois une base historique fidèle et un renouvellement continu de ses membres.

In [ ]:
years = np.arange(1994, 2027)

adhesion_counts = (
    data["GROUPE - Année adhésion *"]
    .dropna()
    .astype(int)
    .value_counts()
    .reindex(years, fill_value=0)
    .sort_index()
)

demission_counts = (
    data["GROUPE - Année démission *"]
    .dropna()
    .astype(int)
    .value_counts()
    .reindex(years, fill_value=0)
    .sort_index()
)

# ============================================================
# GRAPHIQUE COMPLET : 1994-2026
# ============================================================

fig, ax = plt.subplots(figsize=(9.84, 4.34), dpi=100)

width = 0.4

ax.bar(
    years - width / 2,
    adhesion_counts,
    width=width,
    label="Adhésions",
    color="#533fe4"
)

ax.bar(
    years + width / 2,
    demission_counts,
    width=width,
    label="Démissions",
    color="#ff5a36"
)

ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'entreprises")

ax.set_title(
    "Évolution des adhésions et des départs",
    fontsize=14,
    fontweight="bold"
)

ax.set_xticks(years)
ax.set_xticklabels(years, rotation=90)

ax.set_xlim(1993.5, 2026.5)

ax.legend(frameon=False)

# Quadrillage horizontal uniquement
ax.grid(axis="x", visible=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


# ============================================================
# ZOOM : 2019-2025
# ============================================================

zoom_years = np.arange(2019, 2026)

fig, ax = plt.subplots(figsize=(9.84, 4.34), dpi=100)

ax.bar(
    zoom_years - width / 2,
    adhesion_counts.loc[zoom_years],
    width=width,
    label="Adhésions",
    color="#533fe4"
)

ax.bar(
    zoom_years + width / 2,
    demission_counts.loc[zoom_years],
    width=width,
    label="Démissions",
    color="#ff5a36"
)

ax.set_xlabel("Année")
ax.set_ylabel("Nombre d'entreprises")

ax.set_title(
    "Évolution des adhésions et des départs — 2019-2025",
    fontsize=14,
    fontweight="bold"
)

ax.set_xticks(zoom_years)
ax.set_xticklabels(zoom_years)

ax.set_xlim(2018.5, 2025.5)

ax.legend(frameon=False)

# Quadrillage horizontal uniquement
ax.grid(axis="x", visible=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

Parmis ceux qui sont encore adhérents (= sans "GROUPE - Année démission *"), combien ont déjà démissioné (var'a_deja_demissionne')

In [ ]:
# Filtrer les entreprises actuellement adhérentes (sans année de démission)
currently_members = data[data['GROUPE - Année démission *'].isna()].copy()

# Analyser la variable 'a_deja_demissionne' pour ces entreprises
analyse_univariee(currently_members, 'a_deja_demissionne')

# Créer une visualisation
fig, ax = plt.subplots(figsize=(10, 6))

demission_counts = currently_members['a_deja_demissionne'].value_counts()
demission_labels = ['Jamais démissionné', 'A déjà démissionné']
demission_data = [demission_counts.get(False, 0), demission_counts.get(True, 0)]

colors_demission = ['#2ecc71', '#e74c3c']
bars = ax.bar(demission_labels, demission_data, color=colors_demission, edgecolor='black', width=0.6)

# Ajouter les labels avec pourcentages
total = sum(demission_data)
for bar, count in zip(bars, demission_data):
    height = bar.get_height()
    percentage = (count / total) * 100
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}\n({percentage:.1f}%)',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Nombre d\'entreprises', fontsize=12)
ax.set_title('Parmi les entreprises actuellement adhérentes:\nCombien ont déjà démissionné?', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(demission_data) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal d'entreprises actuellement adhérentes: {total}")
print(f"Jamais démissionné: {demission_data[0]} ({demission_data[0]/total*100:.1f}%)")
print(f"A déjà démissionné: {demission_data[1]} ({demission_data[1]/total*100:.1f}%)")

In [ ]:
# Afficher les 17 entreprises qui ont déjà démissionné parmi les actuellement adhérentes
formerly_resigned = currently_members[currently_members['a_deja_demissionne'] == True][
    ['GROUPE - Nom', 'GROUPE - Année adhésion *', 'GROUPE - Année démission *', 
     'duree_derniere_adhesion', 'nb_contacts', 'sector_group']
].sort_values('duree_derniere_adhesion', ascending=False)

print(f"Les {len(formerly_resigned)} entreprises actuellement adhérentes qui ont déjà démissionné:\n")
print(formerly_resigned.to_string())

In [ ]:
# Créer un diagramme circulaire des secteurs des 17 entreprises qui ont déjà démissionné
sector_counts_resigned = formerly_resigned['sector_group'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.Set3(range(len(sector_counts_resigned)))

wedges, texts, autotexts = ax.pie(
    sector_counts_resigned,
    labels=sector_counts_resigned.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=colors,
    wedgeprops={"edgecolor": "white", "linewidth": 2}
)

# Améliorer la lisibilité des labels
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(10)

ax.set_title("Répartition sectorielle des 17 entreprises\nqui ont déjà démissionné", 
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

# Afficher également le détail
print("\nRépartition des secteurs parmi les 17 entreprises ayant démissionné:")
print(sector_counts_resigned)

In [ ]:
analyse_univariee(data, "Organisation - Membership status")

### Durée de la dernière adhésion

In [ ]:
import numpy as np

import matplotlib.pyplot as plt

series = data["duree_derniere_adhesion"].dropna().astype(int)

fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.arange(series.min() - 0.5, series.max() + 1.5, 1)
counts, _, patches = ax.hist(series, bins=bins, color='#533fe4')

ax.set_xlabel("Durée de la dernière adhésion (années)")
ax.set_ylabel("Nombre d'entreprises")
ax.set_title(
    "Durée de la dernière adhésion",
    fontsize=14,
    fontweight="bold"
)
ax.set_xticks(np.arange(series.min(), series.max() + 1, 1))
# Quadrillage horizontal uniquement
ax.grid(axis="x", visible=False)
ax.grid(axis="y", linestyle="--", alpha=0.4)

# Suppression des bordures supérieure et droite
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

La variable duree_derniere_adhesion mesure le nombre d'années écoulées depuis la dernière adhésion de l'entreprise.  

Les statistiques descriptives montrent :
- moyenne : 14,7 ans
- médiane : 11 ans
- minimum : 0 an
- premier quartile : 4 ans
- troisième quartile : 27 ans
- maximum : 32 ans  

La durée d'adhésion apparaît très hétérogène au sein de la population étudiée.  

La moitié des entreprises ont une ancienneté inférieure à 11 ans, tandis qu'un quart des organisations affichent une ancienneté supérieure à 27 ans. Cette dispersion importante illustre la coexistence d'entreprises récemment engagées aux côtés d'acteurs historiquement liés à l'UDM.  

L'écart entre la moyenne (14,7 ans) et la médiane (11 ans) suggère également la présence d'un nombre significatif d'entreprises très anciennes qui tirent la moyenne vers le haut.  

L'UDM semble s'appuyer sur un socle solide d'entreprises fidèles depuis de nombreuses années tout en continuant à intégrer régulièrement de nouveaux membres.  

### Nombre d'adhésions au cours de l'historique

In [ ]:
analyse_univariee(data, "GROUPE - Nombre d'adhésion")

La majorité des organisations n'ont connu qu'une seule adhésion au cours de leur historique :
- 1 adhésion : 81,4 %
- 2 adhésions : 15,8 %
- 3 adhésions : 2,8 %  

Ce résultat met en évidence une forte stabilité des parcours d'adhésion.  

Pour plus de 8 entreprises sur 10, l'historique ne comporte qu'une seule période d'adhésion. Les situations de départ puis de réadhésion restent relativement marginales.  

Cela suggère que la relation entre l'UDM et ses adhérents est généralement construite dans la durée plutôt qu'au travers d'adhésions intermittentes.

## 1.3 Structure organisationnelle
Afin de mieux caractériser les entreprises présentes dans la base, plusieurs variables décrivant les niveaux hiérarchiques représentés ont été étudiées.

### Niveau hiérarchique

In [ ]:
cols = ["has_C_level", "has_M_level", "has_F_level"]

status = data[cols].apply(
    lambda s: s.map({1.0: "Présent", 0.0: "Absent"}).fillna("Manquant")
)

# Effectifs
counts = status.apply(pd.Series.value_counts).T.fillna(0)[["Présent", "Absent", "Manquant"]]

# Conversion en pourcentages
percentages = counts.div(counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 5))
percentages.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=["#4c72b0", "#55a868", "#c44e52"]
)

ax.set_title("Répartition des niveaux hiérarchiques identifiés")
ax.set_xlabel("Variable")
ax.set_ylabel("Pourcentage d'entreprises (%)")
ax.set_xticklabels(["C", "M", "F"], rotation=0)
ax.set_ylim(0, 100)

# Afficher les pourcentages dans les barres
for container in ax.containers:
    labels = [
        f"{v:.1f}%" if v > 0 else ""
        for v in container.datavalues
    ]
    ax.bar_label(container, labels=labels, label_type="center")

plt.tight_layout()
plt.show()

In [ ]:
cols = ['bool_Is C LEVEL', 'bool_Is F LEVEL', 'bool_Is M LEVEL']

level_counts = contacts[cols].eq(1.0).sum()

# Réordonner dans l'ordre souhaité : C, F, M
level_counts = level_counts.reindex(['bool_Is C LEVEL', 'bool_Is F LEVEL', 'bool_Is M LEVEL'])
labels = ["C level", "F level", "M level"]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, level_counts.values, color=["#4c72b0", "#c44e52", "#55a868"], edgecolor='black', width=0.6)

# Ajouter les effectifs et pourcentages sur les barres
total_contacts = len(contacts)
for bar, count in zip(bars, level_counts.values):
    height = bar.get_height()
    percentage = (count / total_contacts) * 100
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}\n({percentage:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Nombre de contacts', fontsize=12)
ax.set_xlabel('Niveau hiérarchique', fontsize=12)
ax.set_title('Répartition des niveaux hiérarchiques dans la base membres', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(level_counts.values) * 1.15)
ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nTotal de contacts: {total_contacts}")
for label, count in zip(labels, level_counts.values):
    print(f"{label}: {int(count)} ({count/total_contacts*100:.1f}%)")

### Présence de VIP

In [ ]:
analyse_univariee(data, "has_VIP")

Parmi les 177 entreprises analysées, 50 disposent d'au moins un contact identifié comme VIP, soit 28 % de la population, contre 72 % qui n'en possèdent pas.  

La présence d'un contact VIP reste relativement limitée au sein de la base. Seule une entreprise sur quatre environ bénéficie de ce type de relation privilégiée avec l'UDM.  

Cela suggère que le statut VIP n'est pas une caractéristique généralisée mais concerne un sous-ensemble spécifique d'organisations.  

## 1.4 Présence des différents départements
Les variables de département indiquent si l'UDM dispose d'au moins un contact appartenant à une fonction donnée au sein de l'entreprise. Une même organisation peut être représentée dans plusieurs départements simultanément ; les pourcentages présentés ne sont donc pas exclusifs les uns des autres.

In [ ]:
departement =[
    'department_RH',
    'department_Marketing',
    'department_Direction_générale',
    'department_Juridique',
    'department_Insights',
    'department_Médias',
    'department_Impact',
    'department_Affaires_Publiques',
    'department_Performance_digitale'
]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

dept_rates = data[departement].mean() * 100
dept_rates = dept_rates.sort_values(ascending=False)


fig, ax = plt.subplots(figsize=(10, 9))

colors = plt.cm.viridis(np.linspace(0, 1, len(dept_rates)))
dept_rates.plot(kind="bar", color=colors, ax=ax)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

ax.set_ylabel("Taux de présence (%)")
ax.set_xlabel("Département")
ax.set_title("Taux de présence des variables de département")
ax.set_ylim(0, 100)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.set_xticklabels(
    [col.replace("department_", "") for col in dept_rates.index],
    rotation=45,
    ha="right"
)

plt.tight_layout()
plt.show()

Le département Marketing est de très loin le plus représenté, avec au moins un contact identifié dans 96 % des entreprises.  
Les autres fonctions les plus fréquemment représentées sont :
- Performance digitale 
- Juridique 
- Médias 
- Direction générale 
- Insights 

À l'inverse, certaines fonctions sont moins souvent présentes :
- Impact 
- Affaires publiques 
- RH 

Ces résultats sont cohérents avec la mission et le positionnement de l'UDM.
La quasi-totalité des entreprises disposent d'au moins un contact Marketing, ce qui confirme que cette fonction constitue le principal point d'entrée de l'association auprès de ses adhérents. Le Marketing apparaît ainsi comme le cœur de la relation entre les entreprises et l'UDM.  

Au-delà du Marketing, la forte présence des fonctions Juridique, Performance Digitale, Médias et Direction Générale suggère que les sujets portés par l'UDM mobilisent un large éventail d'expertises. Les enjeux traités ne semblent pas se limiter aux problématiques marketing traditionnelles mais concernent également les dimensions réglementaires, digitales et stratégiques.  

La présence de contacts de Direction Générale dans plus d'une entreprise sur deux est particulièrement intéressante. Elle montre que les sujets suivis par l'UDM atteignent fréquemment un niveau décisionnel élevé au sein des organisations.  

À l'inverse, les fonctions RH, Affaires Publiques ou Impact sont moins fréquemment représentées.  

Cette analyse met en évidence une relation souvent multi-départementale entre l'UDM et les entreprises.  

L'association n'est pas uniquement en contact avec les équipes marketing mais interagit également avec des fonctions stratégiques, juridiques et spécialisées. Cette diversité d'interlocuteurs peut être interprétée comme un indicateur de pénétration de l'UDM au sein de l'organisation.  

Dans les analyses suivantes, il sera intéressant de vérifier si les entreprises disposant d'une représentation plus large des départements sont également celles qui présentent la plus forte ancienneté d'adhésion.

In [ ]:
nb_departements_representes = data[departement].sum(axis=1)

# Histogramme du nombre de départements représentés
series = nb_departements_representes.dropna().astype(int)

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.arange(series.min() - 0.5, series.max() + 1.5, 1)
counts, _, patches = ax.hist(series, bins=bins, color='C0', edgecolor='black')

ax.set_xlabel("Nombre de départements représentés")
ax.set_ylabel("Nombre d'entreprises")
ax.set_title("Histogramme du nombre de départements représentés")
ax.set_xticks(np.arange(series.min(), series.max() + 1))

for patch, count in zip(patches, counts):
    x = patch.get_x() + patch.get_width() / 2
    ax.annotate(int(count), (x, count), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 1.5 Nombre de contacts identifiés par entreprise

In [ ]:
analyse_univariee(data, "nb_contacts")

Le nombre de contacts varie fortement d'une entreprise à l'autre.  

La moitié des entreprises disposent de moins de 24 contacts, tandis qu'un quart en possède plus de 52. Certaines organisations atteignent même plus de 250 contacts référencés, ce qui témoigne d'une très forte hétérogénéité dans la richesse de l'information disponible.  

L'écart important entre la moyenne (38) et la médiane (24) suggère la présence de quelques entreprises disposant d'un très grand nombre de contacts, qui tirent la moyenne vers le haut. La distribution apparaît donc fortement asymétrique.  

Au-delà d'un simple indicateur de taille de base de données, le nombre de contacts peut être vu comme un indicateur de la profondeur de la relation entre l'entreprise et l'UDM.  

Une entreprise pour laquelle l'UDM connaît plusieurs dizaines de contacts est potentiellement :
- davantage intégrée dans l'écosystème de l'association ;
- présente sur plusieurs événements ou instances ;
- représentée à plusieurs niveaux hiérarchiques ;
- plus facile à réengager en cas de départ d'un interlocuteur.  

À l'inverse, les entreprises disposant d'un très faible nombre de contacts peuvent être plus dépendantes d'une relation individuelle unique.  

Le nombre de contacts ne doit pas être interprété uniquement comme un indicateur d'engagement. Il peut également refléter :
- la taille de l'entreprise ;
- l'ancienneté de la relation avec l'UDM ;

### Groupes avec le plus grand nombre de contacts

In [ ]:
top_groups = (
    data.groupby("GROUPE - Nom", dropna=False)["nb_contacts"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
    .rename(columns={"nb_contacts": "nb_contacts_total"})
)

print(top_groups)

## 1.6 Indicateurs temporels des contacts

Pour les indicateurs temporelles des contacts, il n'est pas possible d'avoir d'indication sur les contacts les plus ancien car la seule indication est la date de création de la fiche contact dans le CRM procurious datant de 2019. Une grande partie des contacts étaient déjà présents avant 2019, cependant on ne saura pas de quand ils datent, ils auront tous la même date de création (le jour du changement de CRM). C'est pourquoi nous pouvons éventuellement nous concentrer sur les contacts plus récemment utilisés mais pas sur les anciens.
Nous traitons alors uniquement le contact le plus récemment inscrit dans le CRM pour chaque entreprise.

In [ ]:
analyse_univariee(data,'youngest_contact_year')

Pour ceux de 2019 c'est même potentiellement plus vieux car les plus ancien auront la date de création de ce CRM (2019) comme date.

In [ ]:
analyse_univariee(data,'days_since_youngest_contact')

In [ ]:
cols = ["GROUPE - Nom", "days_since_youngest_contact", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion", "target"]
sorted_mails = data.sort_values("days_since_youngest_contact", ascending=True).dropna(subset=["days_since_youngest_contact"])[cols]


print("5 groupes avec le plus de days_since_youngest_contact")
display(sorted_mails.tail(5))

## 1.7 Budget marketing

In [ ]:
analyse_univariee(data, "Organisation - Marketing Budget 2019")

# 2. Comment les membres interagissent avec l'UDM ?
Ici on veut voir leur engagement, leurs activités au sein de l'UDM.

In [ ]:
data_with_target = data[data["target"].notna()]
print(f"Après filtrage : {data_with_target.shape[0]:,} lignes × {data_with_target.shape[1]} colonnes")

## 2.1 Comment nos adhérents interagissent-ils avec nos emails ?
Après avoir dressé le portrait de qui sont nos adhérents, cette section s'intéresse à leur comportement vis-à-vis de nos communications par email — le canal le plus systématique et le plus mesurable de la relation entre l'UdM et ses membres. Chaque envoi, ouverture et clic est enregistré au niveau du membre (contact individuel), puis agrégé au niveau de l'adhérent (entreprise / groupe) pour donner une vision consolidée de l'engagement.  

L'analyse couvre quatre dimensions complémentaires :
- le volume : combien de sollicitations un groupe reçoit-il ?
- la lecture : parmi ces sollicitations, combien sont effectivement ouvertes ?
- l'action : au-delà de l'ouverture, combien de contacts vont jusqu'au clic ?
- la fraicheur : depuis combien de temps le dernier signal d'intérêt a-t-il été observé ?  

Ces quatre dimensions ne racontent pas la même histoire : un groupe peut recevoir beaucoup d'emails sans les lire, ou au contraire être peu sollicité mais extrêmement réactif.  

Une précision méthodologique avant de commencer : ces variables sont des moyennes par contact au sein d'un groupe. Un groupe avec un seul contact très actif et un groupe avec cinquante contacts modérément actifs peuvent afficher la même moyenne — la variable ne dit rien, à elle seule, de la taille de l'équipe qui porte cet engagement. C'est un point à garder en tête dans la lecture de chaque graphique.

### Le volume de sollicitation
Analysons le nombre moyen d'emails reçus par contact au sein du groupe, sur la période observée (2019-2025). Plus la variable `avg_mails_received_per_contact` est élevée, plus le groupe est sollicité.

In [ ]:
analyse_univariee(data_with_target, "avg_mails_received_per_contact")

On observe une distribution très étalée mais sans outliers extrêmes : les valeurs vont de 10,67 à 615,25 emails reçus par contact, pour une moyenne de 305.02 et une médiane de 325,21. Les deux indicateurs sont proches, ce qui signale une distribtuion relativement équilibrée, sans grosse asymétrie.  

In [ ]:
cols = ["GROUPE - Nom", "avg_mails_received_per_contact", "nb_contacts", "nb_contacts_ayant_recu_mail", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("avg_mails_received_per_contact", ascending=True).dropna(subset=["avg_mails_received_per_contact"])[cols]

print("5 groupes avec le moins de avg_mails_received_per_contact")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_mails_received_per_contact")
display(sorted_mails.tail(5))

4 contacts chez Nutrimaine :
- Catherine Hostein : 908 mails
- Armelle Fortas : 754 mails
- Aurélie Hemmer : 684 mails
- Sans nom (Guest, id : 180207): 115 mails

D'après Kévin le chiffre semble normal car Nutrimaine est une petite structure où chacun devait avoir un périmètre assez large et il pense qu'on prenait très large en envoyant tout à tout le monde.

Un grand chiffre sur une petite structure doit signifier que beaucoup de mails sont envoyés potentiellement à une unique personne donc les chiffres sont hauts. Alors que dans une grande structure, chacun s'occupe d'un périmètre et donc ne reçoit pas tous les mails.

In [ ]:
analyse_univariee(data_with_target, "nb_moyen_emails_recus_par_contact_par_an")

In [ ]:
cols = ["GROUPE - Nom", "nb_moyen_emails_recus_par_contact_par_an", "nb_contacts", "nb_contacts_ayant_recu_mail", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("nb_moyen_emails_recus_par_contact_par_an", ascending=True).dropna(subset=["nb_moyen_emails_recus_par_contact_par_an"])[cols]

print("5 groupes avec le moins de nb_moyen_emails_recus_par_contact_par_an")
display(sorted_mails.head(5))

print("5 groupes avec le plus de nb_moyen_emails_recus_par_contact_par_an")
display(sorted_mails.tail(5))

### La lecture des emails
Analyse le nombre moyen d'emails ouverts par contact et le taux d'ouverture.
Cette seconde variable est la plus fiable pour comparer des groupes de tailles et de volumes de sollicitation différents.

In [ ]:
analyse_univariee(data, "avg_mails_open_per_contact")

La distribution va de 0 à 227.68 emails ouverts par contact, avec une moyenne de 55.06 et une médiane de 45.48. La distribution est étalée vers la droite, tirée par quelques gros lecteurs.
On observe que les chiffres sont nettement plus bas que la moyenne des mails reçus par contacts. Cela signifie que de nombreux mails envoyés ne sont jamais ouverts (combien ?)

In [ ]:
analyse_univariee(data_with_target, "avg_taux_ouverture")

Le graphique montre une distribution plus resserrée mais tout aussi asymétrique : moyenne à 20,09%, médiane à 14,93%. La moitié des groupes ouvre donc moins d'un email sur cinq, et un quart des groupes se situe même sous les 9,03% (premier quartile). À l'autre extrémité, le dernier quartile démarre à 27,87% d'ouverture.

In [ ]:
cols = ["GROUPE - Nom", "avg_taux_ouverture", "avg_mails_received_per_contact", "nb_moyen_emails_recus_par_contact_par_an", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("avg_taux_ouverture", ascending=True).dropna(subset=["avg_taux_ouverture"])[cols]

print("5 groupes avec le moins de avg_taux_ouverture")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_taux_ouverture")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data_with_target, "avg_open_delay_hours_by_contact")

In [ ]:
cols = ["GROUPE - Nom", "avg_open_delay_hours_by_contact", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data.sort_values("avg_open_delay_hours_by_contact", ascending=True).dropna(subset=["avg_open_delay_hours_by_contact"])[cols]

print("5 groupes avec le moins de avg_open_delay_hours_by_contact")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_open_delay_hours_by_contact")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data, "avg_days_open_per_contact")

In [ ]:
cols = ["GROUPE - Nom", "avg_days_open_per_contact", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data.sort_values("avg_days_open_per_contact", ascending=True).dropna(subset=["avg_days_open_per_contact"])[cols]

print("5 groupes avec le plus de avg_days_open_per_contact")
display(sorted_mails.tail(5))

### Le passage à l'action
On analyse le nombre moyen d'emails différents sur lesquels un contact a cliqué au moins une fois (`avg_mails_clicked_per_contact`) et le nombre total de clics (`avg_clicks_per_contact`), sans distinguer si ces clics portent sur un seul email ou sur plusieurs — une mesure davantage de l'intensité, qui peut être gonflée par un contact cliquant plusieurs fois sur le même contenu.

In [ ]:
analyse_univariee(data, "avg_mails_clicked_per_contact")

In [ ]:
analyse_univariee(data, "avg_clicks_per_contact")

#### Retirer les valeurs extrêmes

L'écart entre les deux variables est immédiatement visible dans les chiffres : le maximum de avg_mails_clicked_per_contact est de 151,79, tandis que celui de avg_clicks_per_contact grimpe à 4849 — un facteur 30 environ. Cela confirme que la seconde variable est beaucoup plus sensible à des comportements individuels extrêmes (un contact hyperactif qui clique en boucle) qu'à un engagement réparti sur l'équipe du groupe.

In [ ]:
group_clicks = (
    data.groupby("GROUPE - Nom", dropna=False)[["avg_mails_clicked_per_contact", "duree_derniere_adhesion", "avg_mails_received_per_contact", "nb_contacts", "nb_contacts_ayant_recu_mail"]]
    .mean()
    .reset_index()
)

q75 = group_clicks["avg_mails_clicked_per_contact"].quantile(0.75)

group_clicks_last_quartile = group_clicks[
    group_clicks["avg_mails_clicked_per_contact"] >= q75
].sort_values(by="avg_mails_clicked_per_contact", ascending=False)

print(f"Seuil du dernier quartile : {q75:.2f}")
group_clicks_last_quartile

In [ ]:
group_clicks = (
    data.groupby("GROUPE - Nom", dropna=False)[["avg_clicks_per_contact", "duree_derniere_adhesion"]]
    .mean()
    .reset_index()
    .rename(columns={"avg_clicks_per_contact": "avg_clicks_per_contact_mean"})
)

q75 = group_clicks["avg_clicks_per_contact_mean"].quantile(0.75)

group_clicks_last_quartile = group_clicks[
    group_clicks["avg_clicks_per_contact_mean"] >= q75
].sort_values(by="avg_clicks_per_contact_mean", ascending=False)

print(f"Seuil du dernier quartile : {q75:.2f}")
group_clicks_last_quartile

BACARDI-MARTINI FRANCE domine très largement avg_clicks_per_contact avec 4849 clics par contact en moyenne — soit près de 2,5 fois le deuxième (ROCHE SAS, 1960,54).

#### Focus sur Bacardi

In [ ]:
bacardi_contacts = contacts[contacts["GROUPE - Nom"] == "BACARDI-MARTINI FRANCE"]

print(f"Nombre de contacts pour ce groupe : {len(bacardi_contacts)}")

y_col = "clicks_count_by_relation"

if y_col not in bacardi_contacts.columns:
    raise ValueError(f"La colonne {y_col!r} est absente de `contacts` pour ce groupe.")


x = np.arange(len(bacardi_contacts))
x_label = "Index du contact"
y = bacardi_contacts[y_col]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, alpha=0.7, edgecolor="k")

ax.set_title(f"Nuage de points")
ax.set_xlabel(x_label)
ax.set_ylabel(y_col)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()



In [ ]:
analyse_univariee(data_with_target, "avg_taux_click")

In [ ]:
cols = ["GROUPE - Nom", "avg_taux_click", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("avg_taux_click", ascending=True).dropna(subset=["avg_taux_click"])[cols]

print("5 groupes avec le moins de avg_taux_click")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_taux_click")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data_with_target, "avg_taux_click_sur_overture")

In [ ]:
cols = ["GROUPE - Nom", "avg_taux_click_sur_overture", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("avg_taux_click_sur_overture", ascending=True).dropna(subset=["avg_taux_click_sur_overture"])[cols]

print("5 groupes avec le moins de avg_taux_click_sur_overture")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_taux_click_sur_overture")
display(sorted_mails.tail(5))

### La fraîcheur de la relation - variables de récence
Quatre variables mesurent l'ancienneté du dernier signal d'engagement (en jours écoulés) : ``avg_days_since_last_open`` et ``avg_days_since_last_click`` (moyennes par contact du délai depuis leur dernière action), ainsi que ``days_since_most_recent_open`` et ``days_since_most_recent_click`` (délai depuis la toute dernière action, tous contacts confondus, au niveau du groupe).


In [ ]:
analyse_univariee(data_with_target, "avg_days_since_last_open")

In [ ]:
analyse_univariee(data, "avg_days_since_last_click")

In [ ]:
analyse_univariee(data, "days_since_most_recent_click")

In [ ]:
analyse_univariee(data_with_target, "days_since_most_recent_open")

In [ ]:
cols = ["GROUPE - Nom", "days_since_most_recent_open", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion", "target"]
sorted_mails = data_with_target.sort_values("days_since_most_recent_open", ascending=True).dropna(subset=["days_since_most_recent_open"])[cols]


print("5 groupes avec le plus de days_since_most_recent_open")
display(sorted_mails.tail(5))

In [ ]:
import seaborn as sns
from scipy.stats import mannwhitneyu

import matplotlib.pyplot as plt

# Préparer les données
df_viz = data_with_target.dropna(subset=["days_since_most_recent_open", "target"]).copy()

# Créer une figure avec plusieurs visualisations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Boxplot : distribution de days_since_most_recent_open par target
sns.boxplot(data=df_viz, x="target", y="days_since_most_recent_open", ax=axes[0], palette="Set2")
axes[0].set_title("Distribution de days_since_most_recent_open par target")
axes[0].set_xlabel("Target (0=Non, 1=Oui)")
axes[0].set_ylabel("Jours depuis l'ouverture la plus récente")
axes[0].grid(alpha=0.3)

# 2. Violin plot pour voir la densité
sns.violinplot(data=df_viz, x="target", y="days_since_most_recent_open", ax=axes[1], palette="Set2")
axes[1].set_title("Densité de days_since_most_recent_open par target")
axes[1].set_xlabel("Target (0=Non, 1=Oui)")
axes[1].set_ylabel("Jours depuis l'ouverture la plus récente")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Statistiques descriptives
print("Statistiques par groupe target:")
print(df_viz.groupby("target")["days_since_most_recent_open"].describe())

# Test statistique (Mann-Whitney U test)
group_0 = df_viz[df_viz["target"] == 0]["days_since_most_recent_open"].dropna()
group_1 = df_viz[df_viz["target"] == 1]["days_since_most_recent_open"].dropna()
statistic, p_value = mannwhitneyu(group_0, group_1)
print(f"\nTest Mann-Whitney U: statistic={statistic:.2f}, p-value={p_value:.4f}")

In [ ]:
# On filtre les lignes avec des valeurs manquantes
df_plot = data_with_target.dropna(
    subset=[
        "avg_days_since_last_open",
        "days_since_most_recent_open",
        "nb_contacts",
    ]
).copy()

# 2. Normalisation de la taille des bulles pour éviter les bulles géantes
# (Échelle entre 20 et 500 px)
min_size, max_size = 20, 500
nb_min, nb_max = df_plot["nb_contacts"].min(), df_plot["nb_contacts"].max()
df_plot["bubble_size"] = min_size + (df_plot["nb_contacts"] - nb_min) / (
    nb_max - nb_min + 1e-5
) * (max_size - min_size)

# 3. Création de la figure
plt.figure(figsize=(10, 7), dpi=100)

scatter = plt.scatter(
    df_plot["days_since_most_recent_open"],
    df_plot["avg_days_since_last_open"],
    s=df_plot["bubble_size"],       # taille des bulles
    cmap="viridis",
    edgecolors="w",  # Contour blanc pour bien séparer les bulles superposées
    linewidth=0.5,
    alpha=0.7
)

plt.xlabel("Jours depuis l'ouverture la plus récente")
plt.ylabel("Jours depuis la dernière ouverture (moyenne)")
plt.title("Fraîcheur de la relation par groupe")

# Diagonale y = x
plt.axline(
    (0, 0),
    slope=1,
    color="black",
    linestyle=":",
    linewidth=1.5,
    alpha=0.7,
    label="Diagonale (y = x)",
)

# Lignes de référence (à adapter)
plt.axvline(30, color="grey", linestyle="--", alpha=0.5)
plt.axhline(90, color="grey", linestyle="--", alpha=0.5)

plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2.2 Comment nos adhérents interagissent-ils avec nos évènements ?

Objectif : Comprendre comment les adhérents interagissent avec les événements : fréquence de participation, réactivité aux invitations, engagement récent et comportement face aux inscriptions.

### Variables de récence

In [ ]:
analyse_univariee(data_with_target, "avg_days_since_last_registration")

In [ ]:
cols = ["GROUPE - Nom", "avg_days_since_last_registration", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion", "target"]
sorted_mails = data_with_target.sort_values("avg_days_since_last_registration", ascending=True).dropna(subset=["avg_days_since_last_registration"])[cols]


print("5 groupes avec le plus de avg_days_since_last_registration")
display(sorted_mails.tail(15))

In [ ]:
analyse_univariee(data_with_target, "days_since_most_recent_registration")

In [ ]:
cols = ["GROUPE - Nom", "days_since_most_recent_registration", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion", "target"]
sorted_mails = data_with_target.sort_values("days_since_most_recent_registration", ascending=True).dropna(subset=["days_since_most_recent_registration"])[cols]


print("5 groupes avec le plus de days_since_most_recent_registration")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data, "avg_days_since_last_participation")

In [ ]:
analyse_univariee(data, "days_since_most_recent_participation")

### Variables de fréquence

In [ ]:
analyse_univariee(data_with_target, "avg_registered_count_per_contact")

In [ ]:
cols = ["GROUPE - Nom", "avg_registered_count_per_contact", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("avg_registered_count_per_contact", ascending=True).dropna(subset=["avg_registered_count_per_contact"])[cols]

print("5 groupes avec le moins de avg_registered_count_per_contact")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_registered_count_per_contact")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data_with_target, "registered_count_by_organisation")

In [ ]:
analyse_univariee(data_with_target, "nb_inscriptions_par_organisation_par_an")

In [ ]:
import matplotlib.pyplot as plt
from adjustText import adjust_text
import matplotlib.pyplot as plt

df_scatter = data_with_target.dropna(
    subset=["nb_inscriptions_par_organisation_par_an", "nb_contacts"]
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    df_scatter["nb_contacts"],
    df_scatter["nb_inscriptions_par_organisation_par_an"],
    alpha=0.6,
    edgecolor="k",
    linewidth=0.5,
)

# Seuils (à adapter)
seuil_contacts = df_scatter["nb_contacts"].quantile(0.95)
seuil_inscriptions = df_scatter["nb_inscriptions_par_organisation_par_an"].quantile(0.95)

outliers = df_scatter[
    (df_scatter["nb_contacts"] >= seuil_contacts)
    | (df_scatter["nb_inscriptions_par_organisation_par_an"] >= seuil_inscriptions)
]

texts = []

for _, row in outliers.iterrows():
    texts.append(
        ax.text(
            row["nb_contacts"],
            row["nb_inscriptions_par_organisation_par_an"],
            row["GROUPE - Nom"],
            fontsize=8,
        )
    )

adjust_text(
    texts,
    ax=ax,
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.5)
)

ax.set_xlabel("nb_contacts")
ax.set_ylabel("nb_inscriptions_par_organisation_par_an")
ax.set_title(
    "Nuage de points : nb_inscriptions_par_organisation_par_an vs nb_contacts"
)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
analyse_univariee(data_with_target, "nb_moyen_inscriptions_par_contact_par_an")

In [ ]:
cols = ["GROUPE - Nom", "nb_moyen_inscriptions_par_contact_par_an", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data_with_target.sort_values("nb_moyen_inscriptions_par_contact_par_an", ascending=True).dropna(subset=["nb_moyen_inscriptions_par_contact_par_an"])[cols]

print("5 groupes avec le moins de nb_moyen_inscriptions_par_contact_par_an")
display(sorted_mails.head(5))

print("5 groupes avec le plus de nb_moyen_inscriptions_par_contact_par_an")
display(sorted_mails.tail(5))

In [ ]:
analyse_univariee(data, "avg_present_count_per_contact")

In [ ]:
cols = ["GROUPE - Nom", "avg_present_count_per_contact", "avg_mails_received_per_contact", "nb_contacts", "duree_derniere_adhesion"]
sorted_mails = data.sort_values("avg_present_count_per_contact", ascending=True).dropna(subset=["avg_present_count_per_contact"])[cols]

print("5 groupes avec le moins de avg_present_count_per_contact")
display(sorted_mails.head(5))

print("5 groupes avec le plus de avg_present_count_per_contact")
display(sorted_mails.tail(5))

### Variables de taux d'engagement
Quand un adhérent est sollicité, quelle est sa propension à participer ?

In [ ]:
analyse_univariee(data, "avg_presence_rate_by_relation")

In [ ]:
analyse_univariee(data, "avg_spontaneous_participation_rate_by_relation")

In [ ]:
analyse_univariee(data, "avg_invitation_reactivity_by_relation")

### Variables d'anticipation
Combien de temps à l'avance les adhérents s'inscrivent-ils à un évènement?

In [ ]:
analyse_univariee(data, "average_anticipation_days_by_group")

## 2.3 Appétence à recevoir les communications

L'engagement des entreprises envers l'UDM ne se limite pas à leur adhésion. Il se matérialise également à travers leur comportement vis-à-vis des communications envoyées par l'association.  

Cette section vise à mesurer l'exposition et la réceptivité des entreprises aux différents dispositifs d'information proposés par l'UDM : newsletters, veille juridique, communications partenaires et communautés.  

Trois dimensions complémentaires sont étudiées :
- l'acceptation des communications ;
- la joignabilité effective des contacts ;
- les signaux de désengagement.

### Acceptation des communications UDM

In [ ]:
# boxplots côte à côte pour 4 variables avec palette viridis
cols_plot = [
    'comm_accept_pct_Communication - Newsletters',
    'comm_accept_pct_Communication - Veille juridique',
    'comm_accept_pct_Communication - Partner communications',
    'comm_accept_pct_Communication - Communautés'
]

data_to_plot = [data_with_target[c].dropna() for c in cols_plot]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(data_to_plot)))

bp = ax.boxplot(data_to_plot, patch_artist=True,
                labels=['Newsletters', 'Veille juridique', 'Partner comms', 'Communautés'])

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
for element in ('whiskers', 'caps', 'medians'):
    plt.setp(bp[element], color='black')

ax.set_ylabel("comm_accept_pct (%)")
ax.set_title("Boxplots des taux d'acceptation par type de communication")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
data_with_target[cols_plot].describe().round(2).T

In [ ]:
# Filtrer les contacts qui ont accepté "Communication - Veille juridique"
contacts_veille = contacts[contacts['Communication - Veille juridique'] == 1.0]

# Sélectionner les colonnes department_* et garder seulement celles avec au moins une valeur 1
dept_cols = [col for col in contacts_veille.columns if col.startswith('department_')]
dept_data = contacts_veille[dept_cols]

# Compter les occurrences de chaque département (1 = représenté)
dept_counts = (dept_data == 1.0).sum()
dept_counts = dept_counts[dept_counts > 0].sort_values(ascending=False)

# Renommer les colonnes pour plus de lisibilité (enlever le préfixe 'department_')
dept_labels = [col.replace('department_', '') for col in dept_counts.index]

# Créer un diagramme circulaire
fig, ax = plt.subplots(figsize=(10, 8))
ax.pie(dept_counts.values, labels=dept_labels, autopct='%1.1f%%', startangle=90)
ax.set_title('Départements représentés pour "Communication - Veille juridique"')
plt.tight_layout()
plt.show()

### Taille du public réellement engagé

In [ ]:
# boxplots côte à côte pour 4 variables avec palette viridis
cols_plot = [
    'comm_accept_nb_Communication - Newsletters',
    'comm_accept_nb_Communication - Veille juridique',
    'comm_accept_nb_Communication - Partner communications',
    'comm_accept_nb_Communication - Communautés'
]

data_to_plot = [data[c].dropna() for c in cols_plot]

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(data_to_plot)))

bp = ax.boxplot(data_to_plot, patch_artist=True,
                labels=['Newsletters', 'Veille juridique', 'Partner comms', 'Communautés'])

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
for element in ('whiskers', 'caps', 'medians'):
    plt.setp(bp[element], color='black')

ax.set_ylabel("comm_accept_nb")
ax.set_title("Boxplots d'acceptation par type de communication")
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### Qualité et joignabilité de la base contacts

In [ ]:
analyse_univariee(contacts,"Recipient Status")

Les contacts recoivent à 86% les emails de l'udm.

In [ ]:
cols = ["GROUPE - Nom", "recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)", "target", "nb_contacts", "nb_contacts_ayant_recu_mail", "duree_derniere_adhesion"]
sorted_mails = data.sort_values("recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)", ascending=False).dropna(subset=["recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)"])[cols]

print("5 groupes avec le plus de recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)")
display(sorted_mails.head(5))

In [ ]:
cols = ["GROUPE - Nom", "recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)", "target", "nb_contacts", "nb_contacts_ayant_recu_mail", "duree_derniere_adhesion"]
sorted_mails = data.sort_values("recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)", ascending=False).dropna(subset=["recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)"])[cols]

print("5 groupes avec le plus de recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)")
display(sorted_mails.head(5))

In [ ]:
analyse_univariee(data, 'recipient_status_pct_Inactive, relation will not receive mailings (Reason: Too many bounces)')

In [ ]:
analyse_univariee(data, 'recipient_status_pct_Inactive, relation will not receive mailings (Reason: Manually disabled)')

### Désengagement

In [ ]:
analyse_univariee(data,'nb_unsubscribed_by_relation')

In [ ]:
result = (
    data.loc[
        data['nb_unsubscribed_by_relation'].notna(),
        ['GROUPE - Nom', 'duree_derniere_adhesion', 'nb_contacts', 'nb_unsubscribed_by_relation']
    ]
    .sort_values('nb_unsubscribed_by_relation', ascending=False)
    .head(10)
)
print(result)

In [ ]:
analyse_univariee(data, 'unsub_rate_pct')

In [ ]:
result = (
    data.loc[
        data['unsub_rate_pct'].notna(),
        ['GROUPE - Nom', 'duree_derniere_adhesion', 'nb_contacts', 'unsub_rate_pct']
    ]
    .sort_values('unsub_rate_pct', ascending=False)
    .head(15)
)
print(result)

Les 2 taux à 100% concerne seulement un contact. C'est important à prendre en compte mais à nuancer par rapport à un taux moins élevé mais concernant plus de contacts. Les plus inquiétants sont ceux ayant un taux élevés alors  qu'ils ont un grand nombre de contacts.